# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and exports
trades for a training-selected wallet universe.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [28]:
# TO recalculate: 
# remove token_df file
# remove enhanced trades
# remove processed trades
# remove processed markets

# rm /Users/vobornij/projects/polymarket/data/markets_processed/*.parquet
# rm /Users/vobornij/projects/polymarket/data/trades_polygon_enriched/enriched_*
# rm /Users/vobornij/projects/polymarket/data/polygon_trades_processed/*.parquet

In [29]:
%load_ext autoreload
%autoreload 2

import json
import datetime
from pathlib import Path

import pandas as pd

import numpy as np
np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [30]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2025, 1, 1)
END_DATE   = datetime.date(2027, 4, 5)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-4%-per-shard wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 6, 1)

COPY_WINDOW_SECONDS = 5 * 60  # 5 minutes

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
ENRICHED_TRADES_DIR  = Path("../../data/trades_polygon_enriched")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

The token lookup table (`token_id → condition_id, outcome, token_winner, final_price`) is
built once via `build_token_lookup_df()` and persisted to
`data/markets_processed/token_lookup.parquet`. Subsequent runs load it with
`load_token_lookup_df()` instead of re-parsing all market JSON.

In [31]:
# # Load only the market files whose month falls within [START_DATE, END_DATE].
# import datetime as _dt
# def _file_in_range(p, start, end):
#     """Return True if YYYY-MM.parquet filename falls within the date range."""
#     try:
#         y, m = (int(x) for x in p.stem.split("-"))
#         file_date = _dt.date(y, m, 1)
#         return start.replace(day=1) <= file_date <= end.replace(day=1)
#     except Exception:
#         return False

# market_files = [
#     p for p in sorted(MARKETS_DIR.glob("*.parquet"))
#     if _file_in_range(p, START_DATE, END_DATE)
# ]
# print(f"Found {len(market_files)} market file(s)")

# all_market_rows = pd.concat(
#     [pd.read_parquet(f) for f in market_files],
#     ignore_index=True,
# )
# # deduplicate by condition_id (keep first occurrence)
# all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
# print(f"Unique markets (all):  {len(all_market_rows):,}")

# # ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# # Parse end_date_iso as a date; keep only markets whose resolution date falls
# # within [START_DATE, END_DATE] (inclusive).
# end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
# in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
# all_market_rows = all_market_rows[in_range].reset_index(drop=True)
# print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
# all_market_rows.head(2)

In [32]:
from polymarket_analysis.data.data_catalogue import (
    load_markets_processed,
    build_token_lookup_df,
    load_token_lookup_df,
)
token_df = load_token_lookup_df()
print(f"Token lookup rows: {len(token_df):,}")

token_df = token_df[
    ~(token_df['primary_tag'].isin(['Sports', 'Crypto']))
    & token_df['token_winner'].notna()
    ]

# Per-condition last trade timestamp (across all raw trades globally)
condition_max_ts = []
for f in sorted(TRADES_DIR.glob("*.parquet")):
    df = pd.read_parquet(f, columns=["block_timestamp", "condition_id"])
    condition_max_ts.append(df.groupby("condition_id")["block_timestamp"].max())
condition_max_ts = pd.concat(condition_max_ts).groupby(level=0).max().reset_index()
condition_max_ts.columns = ["condition_id", "block_timestamp"]
condition_max_ts["last_condition_trade_ts"] = pd.to_datetime(condition_max_ts["block_timestamp"], unit="s", utc=True)
token_df = token_df.merge(condition_max_ts[["condition_id", "last_condition_trade_ts"]], on="condition_id", how="left")
print(f"Added last_condition_trade_ts for {len(condition_max_ts):,} conditions")

len(token_df)

Processing 77 market file(s)…
Saved 3,253,330 token rows → /Users/vobornij/projects/polymarket/data/markets_processed/token_lookup.parquet
Token lookup rows: 3,253,330
Added last_condition_trade_ts for 122,327 conditions


407598

## 2 · Process trades — Phase 1: select top-4% wallets per shard

Each shard is processed independently in parallel. Only per-wallet training-period P&L
is returned (no trade rows), so memory usage is minimal. The union of top-4% wallets
across all shards becomes the candidate set for Phase 2.

Selection here is based on **training trade_pnl** (not copyability).


In [33]:
# Preprocess: enrich by copyable quantity if not present

from concurrent.futures import ProcessPoolExecutor, as_completed

from polymarket_analysis.preprocessing.fill_extender import enrich_shard

trade_files = sorted(TRADES_DIR.glob("*.parquet"))

with ProcessPoolExecutor() as executor:
    futures = {executor.submit(enrich_shard, f, ENRICHED_TRADES_DIR, COPY_WINDOW_SECONDS, token_df): f for f in trade_files}

    total = len(futures)

    for i, fut in enumerate(as_completed(futures), start=1):
        f = futures[fut]
        try:
            fut.result()
        except Exception as e:
            print(f"Error processing {f}: {e}")
            continue

        if i % 4 == 0 or i == total:
            print(f"{i}/{total} shards processed")

7348759 trades in 0.parquet
7167197 trades in 1.parquet
5188016 trades after merging with token_df for 0.parquet
7226603 trades in 2.parquet
7598936 trades in 3.parquet
5375465 trades after merging with token_df for 1.parquet
5854004 trades after merging with token_df for 2.parquet
7822533 trades in 4.parquet
5966776 trades after merging with token_df for 3.parquet
7611114 trades in 5.parquet
6152557 trades after merging with token_df for 4.parquet
7626310 trades in 7.parquet
7342974 trades in 8.parquet
5739932 trades after merging with token_df for 5.parquet
8076900 trades in 6.parquet
7167528 trades in 9.parquet
5810588 trades after merging with token_df for 8.parquet
5915049 trades after merging with token_df for 7.parquet
5360581 trades after merging with token_df for 9.parquet
6430552 trades after merging with token_df for 6.parquet
Enriched 0.parquet with copyable_qty
Enriched 1.parquet with copyable_qty
8587020 trades in a.parquet
Enriched 9.parquet with copyable_qty
7862558 tra

In [34]:
# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', 200)

# MARKET = '0x02d03a859b9d64c7cbbc2a2c3172898bb89b379dba10d54d74ee2c1d5036a71b'

# df = pd.read_parquet(ENRICHED_TRADES_DIR / "enriched_0.parquet")
# df = df[df['condition_id'] == MARKET]
# df[['side', 'price', 'quantity', 'ts', 'token_id', 'tx_hash', 'wallet', 'position']].head(5)

In [35]:
# len(df)
# df[df['tx_hash']=='0xffece5c5cde2b0e1b9375cf30cbb3af09f087967143aff3b4fe5e53c8d1b58c3']

In [36]:
# df['ts'] = pd.to_datetime(df['block_timestamp'], unit='s')
# df['token_id'] = df['token_id'].str[:5]
# df['tx_hash'] = df['tx_hash'].str[:5]
# df[["wallet", 'tx_hash', 'ts', "side", "price", "quantity","token_id", "token_winner", "avail_copy_qty", "avail_copy_total_vol", "avail_copy_count","condition_id"]].head(1)

In [37]:
from polymarket_analysis.preprocessing.shard_processor import select_top_wallets_shard

# ── load token resolution DataFrame ─────────────────────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

enriched_trade_files = sorted(ENRICHED_TRADES_DIR.glob("*.parquet"))

# ── Phase 1: identify top-4% wallets per shard (parallel) ────────────────────
print("Phase 1 — selecting top-4% wallets per shard (trade_pnl)...")
shard_wallet_pnls: dict[str, float] = {}   # wallet → best shard pnl (for diagnostics)
shard_wallet_thresholds: dict[int, float] = {}   # shard index → top-pct threshold (for diagnostics)
total_raw_rows      = 0
total_in_range_rows = 0
total_candidates    = 0

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(
            select_top_wallets_shard,
            f,
            token_df,
            END_TRAIN_TS,
            top_pct=0.04,
            selection_pnl="trade_pnl",
        ): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        wallet_pnl, stats = fut.result()
        total_raw_rows      += stats["raw_rows"]
        total_in_range_rows += stats["in_range_rows"]
        total_candidates    += stats["candidate_wallets"]
        # union: keep the wallet; if it appears in multiple shards take max pnl
        for w, pnl in wallet_pnl.items():
            if w not in shard_wallet_pnls or pnl > shard_wallet_pnls[w]:
                shard_wallet_pnls[w] = pnl
        shard_wallet_thresholds[i] = stats["threshold"]
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(
                f"  {i}/{len(enriched_trade_files)} shards | "
                f"raw: {total_raw_rows:,} | in-range: {total_in_range_rows:,} | "
                f"wallet union so far: {len(shard_wallet_pnls):,}"
            )

top_wallets: set[str] = set(shard_wallet_pnls.keys())
print(f"\nPhase 1 done. Candidate wallets (union of top-4% per shard): {len(top_wallets):,}")
print(f"Threshold pnls per shard: {shard_wallet_thresholds}")


Phase 1 — selecting top-4% wallets per shard (trade_pnl)...
Processing shard enriched_0.parquet...
Processing shard enriched_1.parquet...
Processing shard enriched_2.parquet...
Shard top 4.00% threshold: 81.19 USDC
Processing shard enriched_3.parquet...
Shard top 4.00% threshold: 85.49 USDC
Processing shard enriched_4.parquet...
Shard top 4.00% threshold: 81.72 USDC
Processing shard enriched_5.parquet...
Shard top 4.00% threshold: 114.26 USDC
Processing shard enriched_6.parquet...
Shard top 4.00% threshold: 111.42 USDC
Processing shard enriched_7.parquet...
Shard top 4.00% threshold: 72.53 USDC
  4/16 shards | raw: 22,384,261 | in-range: 22,384,261 | wallet union so far: 29,670
Processing shard enriched_8.parquet...
Shard top 4.00% threshold: 103.37 USDC
Processing shard enriched_9.parquet...
Shard top 4.00% threshold: 85.28 USDC
  8/16 shards | raw: 46,622,351 | in-range: 46,622,351 | wallet union so far: 47,832
Processing shard enriched_a.parquet...
Shard top 4.00% threshold: 90.33 U

## 3 · Phase 2: enrich, group by tx×wallet×side, filter to top wallets

Each shard is processed in parallel.  Fills are inner-joined with settlement data,
aggregated to one row per ``tx_hash × wallet × side``, and filtered to the wallet set
from Phase 1.  Results are concatenated in memory and re-grouped to merge any
transactions that span shard boundaries.

In [38]:
from polymarket_analysis.preprocessing.shard_processor import enrich_and_group_shard

# ── Phase 2: enrich + group + filter (parallel) ──────────────────────────────
print("Phase 2 — grouping and filtering shards...")
shard_results:     list[pd.DataFrame]    = []
wallet_pnl_total:  dict[str, float]      = {}   # accumulated training trade_pnl per wallet
sample_fills = None

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(
            enrich_and_group_shard,
            f,
            token_df,
            END_TRAIN_TS,
            top_wallets,
            wallet_pnl_metric="trade_pnl",
        ): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        grouped_shard, shard_train_pnl = fut.result()
        if not grouped_shard.empty:
            shard_results.append(grouped_shard)
            if sample_fills is None:
                sample_fills = grouped_shard.head(5).copy()
        for w, pnl in shard_train_pnl.items():
            wallet_pnl_total[w] = wallet_pnl_total.get(w, 0.0) + pnl
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(f"  {i}/{len(enriched_trade_files)} shards | shards with data: {len(shard_results)}")

if not shard_results:
    raise ValueError("No in-range trade rows after token filter")

# ── merge cross-shard partial groups ─────────────────────────────────────────
GROUP_KEYS = ["tx_hash", "wallet", "side", "token_id"]
grouped = pd.concat(shard_results, ignore_index=True)
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",               "first"),
        last_condition_trade_ts = ("last_condition_trade_ts", "max"),
        condition_id     = ("condition_id",     "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_x_qty_sum  = ("price_x_qty_sum",  "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
        copyable_pnl     = ("copyable_pnl",     "sum"),
        copyable_qty     = ("copyable_qty",     "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)
grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]
# grouped.drop(columns=["price_x_qty_sum"], inplace=True)
grouped.sort_values(["wallet", "dt"], inplace=True, ignore_index=True)

print(f"\nPhase 2 done.")
print(f"  Grouped rows (all top wallets, all periods): {len(grouped):,}")
print(f"  Unique wallets in grouped:                   {grouped['wallet'].nunique():,}")
grouped.head(5)


Phase 2 — grouping and filtering shards...
  4/16 shards | shards with data: 4
  8/16 shards | shards with data: 8
  12/16 shards | shards with data: 12
  16/16 shards | shards with data: 16

Phase 2 done.
  Grouped rows (all top wallets, all periods): 46,698,607
  Unique wallets in grouped:                   71,081


,tx_hash,wallet,side,token_id,dt,last_condition_trade_ts,condition_id,outcome,token_winner,final_price,position,total_quantity,price_x_qty_sum,trade_value_usdc,final_value_usdc,num_fills,trade_pnl,copyable_pnl,copyable_qty,avail_copy_total_vol,avail_copy_count,avg_price
0,0x0608e4eeb4f28612dfee57a202a790d2f183d4e251f0fcd7dd682685834d9d4d,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,25765184634765339363827807992411986670067641678875992164214223288175352348465,2026-05-18 05:53:53+00:00,2026-05-22 06:11:42+00:00,0x098afead2c41677b0f09ae9d0013ca520eacdb3f0d7c8985c6581de7a0e9ac37,Yes,False,0.00,5000.00,5000.00,700.00,700.00,0.00,1,-700.00,-56.59,404.23,52.55,2.00,0.14
1,0xffb7f006adee358b90369094ef388a7eb54d14d54b3cc37de77c680c4f3462ef,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,28593215534345447362156321620838626774652503878031452558083665828202001699551,2026-05-18 05:54:52+00:00,2026-05-27 03:19:19+00:00,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9f6dc8f59dea51df6c734,Yes,False,0.00,4000.00,4000.00,1316.84,1316.84,0.00,7,-1316.84,-0.61,1.84,0.59,4.00,0.33
2,0x2f6aaa6d6f26ca01bb3326f842fdd7d235b4bc2f8f87b3339afbae215475c339,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,25765184634765339363827807992411986670067641678875992164214223288175352348465,2026-05-19 10:50:26+00:00,2026-05-22 06:11:42+00:00,0x098afead2c41677b0f09ae9d0013ca520eacdb3f0d7c8985c6581de7a0e9ac37,Yes,False,0.00,5500.00,500.00,63.00,63.00,0.00,3,-63.00,-55.60,441.25,78.05,27.00,0.13
3,0xe02e9a4cae3904e6e902aec3a5fd4369b89586e64f487b125d87627d766bc93c,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,SELL,28593215534345447362156321620838626774652503878031452558083665828202001699551,2026-05-23 05:15:15+00:00,2026-05-27 03:19:19+00:00,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9f6dc8f59dea51df6c734,Yes,False,0.00,3917.69,4000.00,3947.01,3947.01,0.00,7,3947.01,1288.37,1305.67,1864.22,23.00,0.99
4,0xe1958851124cdf70de4936b01dc7ec0693e412a1847c9df0b8c8ac3a117a7494,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,52965156717339871692714186660785951310421831919562520539239006775699853846973,2026-05-24 13:03:24+00:00,2026-05-27 06:09:07+00:00,0x421bc1929df1429cf2cb94f80c1ce6a3ed0d1f0b7a2749b9890075f94eb549e9,No,True,1.00,4600.00,4600.00,3036.00,3036.00,4600.00,5,1564.00,157.99,464.68,1397.68,19.00,0.66


## 4 · Phase 3: apply final trade-PnL cut-off and export

Compute the true cross-shard training P&L per wallet. Use the **median** of the
per-shard trade-PnL thresholds from Phase 1 as the export cut-off.


In [39]:
# ── wallet summary (full cross-shard training P&L) ───────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train = grouped[grouped["dt"] < END_TRAIN_TS]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets     = ("condition_id",    "nunique"),
        num_trades      = ("num_fills",       "sum"),
        total_cost_usdc = ("trade_value_usdc", "sum"),
        trade_pnl       = ("trade_pnl",       "sum"),
        copyable_pnl    = ("copyable_pnl",    "sum"),
    )
    .sort_values("trade_pnl", ascending=False)
    .reset_index()
)

# ── PnL cut-off: median of the per-shard trade_pnl thresholds from Phase 1 ───
import statistics
pnl_cutoff = statistics.median(shard_wallet_thresholds.values())

print(f"\nUnique wallets in training data: {wallet_summary['wallet'].nunique():,}")
print(
    f"wallet_summary['trade_pnl'] quantiles:\n"
    f"{wallet_summary['trade_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}"
)

final_wallets = set(
    wallet_summary[
        wallet_summary["trade_pnl"] >= pnl_cutoff
    ]["wallet"]
)

print(f"\nFinal selected wallets (trade-PnL filter): {len(final_wallets):,}")
print(
    f"final_wallets['trade_pnl'] quantiles:\n"
    f"{wallet_summary[wallet_summary['wallet'].isin(final_wallets)]['trade_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}"
)

print(f"Unique wallets after Phase 2 union:    {len(wallet_summary):,}")
print(f"Per-shard trade-PnL median cut-off:    {pnl_cutoff:,.2f} USDC")
print(f"Wallets passing trade-PnL cut-off:     {len(final_wallets):,}")
wallet_summary.head(5)



Unique wallets in training data: 71,081
wallet_summary['trade_pnl'] quantiles:
0.01   -18541.39
0.10    -1215.99
0.20     -303.83
0.30      -27.80
0.40       88.75
0.50      145.59
0.60      228.67
0.70      391.74
0.80      736.97
0.90     2039.70
0.99    34586.64
Name: trade_pnl, dtype: float64

Final selected wallets (trade-PnL filter): 42,949
final_wallets['trade_pnl'] quantiles:
0.01      89.83
0.10     119.17
0.20     156.28
0.30     205.60
0.40     277.98
0.50     386.88
0.60     547.59
0.70     864.22
0.80    1550.04
0.90    4198.21
0.99   55050.51
Name: trade_pnl, dtype: float64
Unique wallets after Phase 2 union:    71,081
Per-shard trade-PnL median cut-off:    86.31 USDC
Wallets passing trade-PnL cut-off:     42,949


,wallet,num_markets,num_trades,total_cost_usdc,trade_pnl,copyable_pnl
0,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,42,2385,3878059.88,1304494.87,501531.92
1,0xc6587b11a2209e46dfe3928b31c5514a8e33b784,259,10495,35705535.39,1226172.83,358080.54
2,0xbaa2bcb5439e985ce4ccf815b4700027d1b92c73,670,65832,23352110.08,1153962.64,334491.88
3,0x0a854897a06d4999e5b2dde5693609f1428ffe9d,51,5449,3450514.00,1075966.68,150921.77
4,0xde7be6d489bce070a959e0cb813128ae659b5f4b,174,13652,7256251.34,948435.27,478858.81


## 5 · Market-level volume summary

In [40]:
# join market metadata (question, end_date) for display
markets_meta = load_markets_processed()[["condition_id", "question", "end_date_iso"]].rename(
    columns={"end_date_iso": "end_date"}
)
grouped_meta = grouped.merge(
    markets_meta,
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Processing 77 market file(s)…
Saved 1,747,674 markets → /Users/vobornij/projects/polymarket/data/markets_processed/markets.parquet
Unique markets: 108,524


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x6d0e09d0f04572d9b1adad84703458b0297bc5603b69dccbde93147ee4443246,US forces enter Iran by April 30?,2026-04-30T00:00:00Z,5709,191240,225470057.41,224421915.59
1,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cdc09e9d47e7d54356c6cd,"US x Iran ceasefire extended by April 22, 2026?",2026-04-21T00:00:00Z,2325,71234,166956381.70,166625320.40
2,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77f981fad98f2bfa0b1bc7,US x Iran ceasefire by April 7?,2026-04-07T00:00:00Z,4762,149680,147091239.79,152661526.86
3,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3a154603dc03c02b57de5,US government shutdown Saturday?,2026-01-31T00:00:00Z,5382,250442,134777265.08,138244099.60
4,0x655e5ca101c466b6293aa15e06173b78b293221803d56e35551f708cd82eb352,Will Zelenskyy wear a suit before July?,2025-06-30T00:00:00Z,2349,72705,131022699.60,131791812.15
5,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,"US x Iran permanent peace deal by June 15, 2026?",2026-06-15T00:00:00Z,3588,195626,122762102.91,148085589.20
6,0xa93b28a6384aefff4e7d5adb2061c444e4a0e95b8ad17755f9cee123c7099b35,"Russia x Ukraine ceasefire by May 31, 2026?",2026-05-31T00:00:00Z,1540,38950,111337293.16,111901448.54
7,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Khamenei out as Supreme Leader of Iran by February 28?,2026-02-28T00:00:00Z,6340,197340,105624591.85,115523192.91
8,0x7cb525e831729325d651017f81cbcb6f1adde5011c7b2283babea00b4ae93ae7,Netanyahu out by March 31?,2026-03-31T00:00:00Z,2897,110679,95111000.90,96116010.14
9,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,"US strikes Iran by February 28, 2026?",2026-01-31T00:00:00Z,7768,199640,73310400.49,90067720.72


## 6 · Wallet statistics (quantiles)

In [41]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")
wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]  = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"] = wallet_stats["num_markets"].round(1)
wallet_stats["copyable_pnl"]    = wallet_stats["copyable_pnl"].round(2)
wallet_stats["trade_pnl"]    = wallet_stats["trade_pnl"].round(2)
wallet_stats

,wallet_count,num_trades,num_markets,copyable_pnl,trade_pnl,wallet_count_complement
quantile,,,,,,
0.00,71,1.00,1.00,-65156.78,-117139.01,71010
0.01,711,1.00,1.00,-11758.80,-18541.39,70370
0.05,3554,2.00,1.00,-2209.24,-3305.70,67527
0.10,7108,4.00,1.00,-876.88,-1215.99,63973
0.25,17770,15.00,3.00,-136.32,-134.64,53311
0.50,35540,58.00,9.00,0.22,145.59,35541
0.75,53311,215.00,29.00,182.63,521.18,17770
0.90,63973,806.00,89.00,750.13,2039.70,7108
0.95,67527,1954.00,189.00,1803.24,5406.41,3554


## 7 · Full enriched trade table

In [42]:
DISPLAY_COLS = [
    "tx_hash", "wallet", "side",
    "dt", "condition_id", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,num_fills
0,0x0608e4eeb4f28612dfee57a202a790d2f183d4e251f0fcd7dd682685834d9d4d,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2026-05-18 05:53:53+00:00,0x098afead2c41677b0f09ae9d0013ca520eacdb3f0d7c8985c6581de7a0e9ac37,Yes,5000.00,5000.00,0.14,False,0.00,700.00,0.00,-700.00,-56.59,1
1,0xffb7f006adee358b90369094ef388a7eb54d14d54b3cc37de77c680c4f3462ef,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2026-05-18 05:54:52+00:00,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9f6dc8f59dea51df6c734,Yes,4000.00,4000.00,0.33,False,0.00,1316.84,0.00,-1316.84,-0.61,7
2,0x2f6aaa6d6f26ca01bb3326f842fdd7d235b4bc2f8f87b3339afbae215475c339,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2026-05-19 10:50:26+00:00,0x098afead2c41677b0f09ae9d0013ca520eacdb3f0d7c8985c6581de7a0e9ac37,Yes,5500.00,500.00,0.13,False,0.00,63.00,0.00,-63.00,-55.60,3
3,0xe02e9a4cae3904e6e902aec3a5fd4369b89586e64f487b125d87627d766bc93c,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,SELL,2026-05-23 05:15:15+00:00,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9f6dc8f59dea51df6c734,Yes,3917.69,4000.00,0.99,False,0.00,3947.01,0.00,3947.01,1288.37,7
4,0xe1958851124cdf70de4936b01dc7ec0693e412a1847c9df0b8c8ac3a117a7494,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2026-05-24 13:03:24+00:00,0x421bc1929df1429cf2cb94f80c1ce6a3ed0d1f0b7a2749b9890075f94eb549e9,No,4600.00,4600.00,0.66,True,1.00,3036.00,4600.00,1564.00,157.99,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46698602,0x070c8061d33e893ea6c921e020c1fc3bd318e437687c5e19be1c3ea3b211eddd,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 20:24:18+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c79963076e524c599d03efda,Yes,200.00,200.00,0.02,False,0.00,4.00,0.00,-4.00,-4.00,1
46698603,0x852015bc27fa54364ba4bd307b710982f8c475e33ff5e2ff1c46933b30c74547,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 23:00:20+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c79963076e524c599d03efda,Yes,4200.00,4000.00,0.00,False,0.00,12.00,0.00,-12.00,-12.00,1
46698604,0x8815e825e712da69beadc37cf8ddc48f2eb2ce9eaec7c859f3fc1bd1ce740b6f,0xffffffe1e093aacd21e4e281e66d543fb0b23455,SELL,2026-02-28 16:39:01+00:00,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Yes,530.00,50.00,0.51,True,1.00,25.35,50.00,-24.65,-24.65,2
46698605,0xdfaa1fa624e9dee67ac0af157cdb94bafec047326147b21625c1a47902742560,0xffffffe1e093aacd21e4e281e66d543fb0b23455,SELL,2026-02-28 20:48:45+00:00,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Yes,0.00,500.00,0.93,True,1.00,463.00,500.00,-37.00,-37.00,1


## 8 · Export: apply trade-PnL cut-off and write partitioned parquet

Filter grouped trades to `final_wallets` (wallets above the median per-shard
trade-PnL threshold), then write one parquet shard per first hex character of
`condition_id` after `0x`.


In [43]:
top_wallets = final_wallets
print(f"Wallets selected for export: {len(top_wallets):,}")

Wallets selected for export: 42,949


In [44]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position",
    "total_quantity", "avg_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl",
    "token_winner", "final_price",
    "last_condition_trade_ts",
    "tx_hash", "num_fills",
    "is_train","copyable_qty", "avail_copy_total_vol", "avail_copy_count",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_df = grouped[grouped["wallet"].isin(top_wallets)].copy()
export_df["is_train"] = export_df["dt"].dt.date <= END_DATE_TRAIN
export_df = export_df[EXPORT_COLS].reset_index(drop=True)

# Partition by the first hex character of condition_id after the "0x" prefix,
# matching the input shard layout (0.parquet … 9.parquet, a.parquet … f.parquet).
export_df["_shard"] = export_df["condition_id"].str[2]

output_files = []
for shard_key, shard_df in export_df.groupby("_shard", sort=True):
    out_file = OUT_DIR / f"{shard_key}.parquet"
    shard_df.drop(columns=["_shard"]).to_parquet(out_file, index=False)
    output_files.append(out_file)

export_df = export_df.drop(columns=["_shard"])
top_trades_preview   = export_df.head(5).copy()
top_trades_count     = len(export_df)
total_top_train_rows = int(export_df["is_train"].sum())

print(f"Total grouped rows exported:  {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
for f in sorted(output_files):
    print(f"  {f.name}  ({pd.read_parquet(f).shape[0]:,} rows)")
print(f"\nSaved partitioned output → {OUT_DIR}")

Total grouped rows exported:  33,384,885
  training rows: 24,366,347
  test rows:     9,018,538
Output shards:  16
  0.parquet  (1,887,504 rows)
  1.parquet  (1,954,996 rows)
  2.parquet  (2,094,095 rows)
  3.parquet  (2,135,147 rows)
  4.parquet  (2,230,151 rows)
  5.parquet  (2,087,199 rows)
  6.parquet  (2,333,361 rows)
  7.parquet  (2,144,402 rows)
  8.parquet  (2,115,565 rows)
  9.parquet  (1,981,371 rows)
  a.parquet  (2,237,069 rows)
  b.parquet  (2,073,759 rows)
  c.parquet  (2,007,524 rows)
  d.parquet  (2,196,646 rows)
  e.parquet  (1,889,800 rows)
  f.parquet  (2,016,296 rows)

Saved partitioned output → ../../data/polygon_trades_processed


In [45]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview

In [46]:
df = pd.read_parquet(ENRICHED_TRADES_DIR/'enriched_3.parquet')
len(df)

5966776

In [47]:
len(df[(df['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20')])

268526

In [48]:
dfs = []
for f in sorted(ENRICHED_TRADES_DIR.glob("*.parquet")):
    dfp = pd.read_parquet(f)
    dfs.append(dfp[dfp['wallet'] == '0x96489abcb9f583d6835c8ef95ffc923d05a86825'])

df_full = pd.concat(dfs, ignore_index=True)

In [49]:
pd.set_option('display.max_columns', None)

In [50]:
enriched = df_full.copy()
enriched["dt"] = pd.to_datetime(enriched["block_timestamp"], unit="s", utc=True)
enriched['final_price'] = enriched['token_winner'] * 1
enriched["final_value_usdc"] = enriched["quantity"] * enriched["final_price"]
enriched["price_x_qty"] = enriched["price"] * enriched["quantity"]

_GROUP_KEYS = ["tx_hash", "wallet", "side"]
# Group fills → one row per tx_hash × wallet × side
grouped = (
    enriched.groupby(_GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",              "first"),
        condition_id     = ("condition_id",    "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("quantity",         "sum"),
        price_x_qty_sum  = ("price_x_qty",     "sum"),
        trade_value_usdc = ("usdc_amount",      "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("log_index",        "count"),
        copyable_qty    = ("copyable_qty",   "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)

grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]

is_buy = grouped["side"] == "BUY"
grouped["trade_pnl"] = np.where(
    is_buy,
    grouped["final_value_usdc"] - grouped["trade_value_usdc"],
    grouped["trade_value_usdc"] - grouped["final_value_usdc"],
)

In [51]:
grouped.groupby('condition_id').agg(
    total_trades = ('num_fills', 'sum'),
    total_volume = ('trade_value_usdc', 'sum'),
    total_pnl    = ('trade_pnl', 'sum'),
).sort_values('total_pnl', ascending=True).head(10)

,total_trades,total_volume,total_pnl
condition_id,,,
0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,2838,1589594.90,-1589594.90
0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77f981fad98f2bfa0b1bc7,1443,745147.28,-631003.04
0x4b02efe53e631ada84681303fd66d79ad615f3d2b6a28b4633d43d935f89af58,1555,585240.20,-577608.94
0xdc4b9ecbfac2c8e98fe18b35c4578a3c64e34ddfe16b1cf8a98ab3ad9d06c088,962,463369.04,-463369.04
0x15aa3c1259a716915e068a0d63c3885d2301d29e8982cbb1717ecb9b63d02d95,857,448182.82,-448182.82
0x70909f0ba8256a89c301da58812ae47203df54957a07c7f8b10235e877ad63c2,1256,477185.87,-419092.55
0x291f0f6023efa933d36cc80fd55f9d176309a92b61b9567fd4f044a8b21873fb,769,409423.70,-409423.70
0x797d586ad45522306490b0cc9b2f21bdf957f3843476fae99f3bcc2cec83b74b,2524,336785.66,-334649.02
0x3fc804e4d893f86d49229eb42492cabf6150f5bcbf36c1b66d1e8f22bc15ec6d,414,285557.54,-285557.54


In [52]:
grouped.groupby(grouped['dt'].dt.date).agg(
    total_trades = ('num_fills', 'sum'),
    total_volume = ('trade_value_usdc', 'sum')
).sort_index(ascending=False).head(10)

,total_trades,total_volume
dt,,
2026-07-05,2,5.78
2026-07-04,2,21.30
2026-07-03,1,63.83
2026-07-02,3,1005.89
2026-07-01,278,227269.63
2026-06-30,449,87826.42
2026-06-29,168,16702.11
2026-06-28,31,7714.55
2026-06-27,212,37676.54


In [53]:
df = pd.read_parquet('/Users/vobornij/projects/polymarket/data/polygon/2026-05-04.parquet')

In [54]:
df.tail()

,block_number,tx_hash,log_index,block_timestamp,order_hash,maker,taker,maker_asset_id,taker_asset_id,protocol_version,side,token_id,maker_amount_filled,taker_amount_filled,fee,builder,metadata
5989041,86409577,0xcec46f9200e900d537f5b39e84b48690972c151c3de4774b5e43801cfea09f4d,5281,1777939198,0x7cd92feca2c9c2aa649cd0c0b34644b70f0ceba49ad9b6b9bd3c33f4685ad70c,0x106a48725a926362bbb137d4c8bccedbaa6aacee,0x33ab02523163fc3e88ddaa381ca1fa05284606d9,0,20315904154374472650661835345885206083613913192457947093594900871269080896805,V2,BUY,20315904154374472650661835345885206083613913192457947093594900871269080896805,10000900,10990000,0,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
5989042,86409577,0xcec46f9200e900d537f5b39e84b48690972c151c3de4774b5e43801cfea09f4d,5283,1777939198,0xf6d86802a56c0d0b6595bcb328576bb958732775eb5ceeac3aad1e6ec44a122f,0x8da42eb2e2c1eba87794e4c30bc689513b90f3d1,0x33ab02523163fc3e88ddaa381ca1fa05284606d9,0,20315904154374472650661835345885206083613913192457947093594900871269080896805,V2,BUY,20315904154374472650661835345885206083613913192457947093594900871269080896805,3649100,4010000,0,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
5989043,86409577,0xcec46f9200e900d537f5b39e84b48690972c151c3de4774b5e43801cfea09f4d,5287,1777939198,0xce23af9a8098a0eb0e69f8ddea7fdd4921d52dd73ce6b3c59c5204a885a7c137,0x33ab02523163fc3e88ddaa381ca1fa05284606d9,0xe111180000d2663c0091e4f400237545b87b996b,0,28348355012943795476930758287039002184648591224808991117129985540950827926854,V2,BUY,28348355012943795476930758287039002184648591224808991117129985540950827926854,4752900,52810000,311410,0x4898df15ec6590495dc6c0fedf951ade3e64001d47f9caf44a64e86fc11959df,0x0000000000000000000000000000000000000000000000000000000000000000
5989044,86409577,0xd66008edc4f19dc239264ebc0687d2e7e85b5183670cd5e9e3e9da1af599bc66,5303,1777939198,0x81f68266bb8d09be6ffce910ea780b506d9ef2540bf34a965f2aaf048e9e38b7,0xae3db1cc155e1b2a5420e256c42cd1649b5a6b95,0x63adda169ca8f3311b361e3e7dc1a98fa882c072,0,28348355012943795476930758287039002184648591224808991117129985540950827926854,V2,BUY,28348355012943795476930758287039002184648591224808991117129985540950827926854,14338260,179228250,0,0x8ba7e2a23e33175c53fccfa30261a35734772d5909e4e4f484f204bfde446c7a,0x0000000000000000000000000000000000000000000000000000000000000000
5989045,86409577,0xd66008edc4f19dc239264ebc0687d2e7e85b5183670cd5e9e3e9da1af599bc66,5307,1777939198,0x012d6a71961ef6008f0bb1e8646c7b74b4a0eca2bf7ebb6358541c0d8f01287c,0x63adda169ca8f3311b361e3e7dc1a98fa882c072,0xe111180000d2663c0091e4f400237545b87b996b,0,20315904154374472650661835345885206083613913192457947093594900871269080896805,V2,BUY,20315904154374472650661835345885206083613913192457947093594900871269080896805,164889990,179228250,949760,0x9fedc0b0702ca6cb294e5321d9491b1e38b8bd2b463a7f7b06df8e6d7553cd18,0x0000000000000000000000000000000000000000000000000000000000000000
